# AttackV2 epsilon accuracy curves

FGSM epsilon strength on x, robust accuracy on y. This notebook reads stored `attack_experiment/tables` aggregate rows and does not re-run attacks.

## 1. Imports, utilities, and config

In [12]:
from pathlib import Path
import sys

try:
    from analytics.paths import attack_experiment_dir, find_project_root
except ModuleNotFoundError:
    repo_root = Path.cwd().resolve()
    while repo_root != repo_root.parent and not (repo_root / "src" / "analytics").is_dir():
        repo_root = repo_root.parent
    sys.path.insert(0, str(repo_root / "src"))
    from analytics.paths import attack_experiment_dir, find_project_root

from analytics.attacks.attacksv2_transfer_tools import (
    AttacksV2Config,
    v2_output_dir,
)
from analytics.attacks.attacksv2_accuracy_curves import (
    ROBUST_METRIC,
    available_curve_groups,
    export_all_accuracy_curve_views,
    load_attack_curve_rows,
    prepare_curve_rows,
)

ROOT = find_project_root()
ATTACK_DIR = attack_experiment_dir(ROOT)
OUT_DIR = v2_output_dir("curves")

CONFIG = AttacksV2Config(
    source_grain="architecture",
    target_grain="architecture",
    comparison_mode="inter_architecture",
    metric=ROBUST_METRIC,
    variant_aggregate_mode="best_robust",
    architecture_aggregate_mode="mean",
    # split_quantum_ansatz_as_architecture=True,
    # blacklist_exact=("Quantum/VQM/some_variant",),
    # blacklist_contains=("HQuanv",),
    # output_dir=None,
)
INCLUDE_SELF_ATTACKS = False

CONFIG

AttacksV2Config(source_grain='architecture', target_grain='architecture', comparison_mode='inter_architecture', metric='Transfer Robust Accuracy', top_k=3, variant_aggregate_mode='best_robust', architecture_aggregate_mode='mean', split_ansatz_as_architecture=False, include_no_entangling_as_ansatz=True, model_blacklist=(), blacklist_contains=(), apply_global_blacklist=True, source_family_filter=None, target_family_filter=None, source_architecture_filter=None, target_architecture_filter=None, source_contains=None, target_contains=None, blacklist_exact=(), split_quantum_ansatz_as_architecture=False, selected_rows=None, selected_columns=None, scatter_display_mode='architectures_only', manual_representatives=(), output_dir=None)

## 2. Crawl and find attack tables

In [13]:
TABLE_FILES = sorted((ATTACK_DIR / "tables").glob("*.parquet")) or sorted((ATTACK_DIR / "tables").glob("*.csv"))
print(f"Attack dir: {ATTACK_DIR}")
print(f"Tables found: {len(TABLE_FILES)}")
TABLE_FILES[:5]

Attack dir: C:\Users\fwilh\Desktop\QB-PIER\logs\ATTACKS\attack_experiment_canonical
Tables found: 68


[WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ATTACKS/attack_experiment_canonical/tables/aggregates_seed=0__source=Classical__CNN__conv_channels=[32,64],_fc_sizes=[128].parquet'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ATTACKS/attack_experiment_canonical/tables/aggregates_seed=0__source=Classical__CNN__conv_channels=[8],_fc_sizes=[10].parquet'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ATTACKS/attack_experiment_canonical/tables/aggregates_seed=0__source=Classical__CNN__conv_channels=[8],_fc_sizes=[32].parquet'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ATTACKS/attack_experiment_canonical/tables/aggregates_seed=0__source=Classical__CNN__LeNet.parquet'),
 WindowsPath('C:/Users/fwilh/Desktop/QB-PIER/logs/ATTACKS/attack_experiment_canonical/tables/aggregates_seed=0__source=Classical__MLP__hidden_sizes=[1024,512,64,64].parquet')]

## 3. Load rows and inspect available groups

In [14]:
raw_df = load_attack_curve_rows(ATTACK_DIR)
print(f"Raw aggregate rows: {raw_df.shape[0]:,} x {raw_df.shape[1]:,}")

groups = available_curve_groups(raw_df, CONFIG, include_self_attacks=INCLUDE_SELF_ATTACKS)
print("Source groups:")
display(groups["source"])
print("Target groups:")
display(groups["target"])

curve_preview = prepare_curve_rows(raw_df, CONFIG, include_self_attacks=INCLUDE_SELF_ATTACKS)
curve_preview.head()

Raw aggregate rows: 48,552 x 13
Source groups:


['Classical/CNN',
 'Classical/MLP',
 'Classical/MLPMixer',
 'Classical/ResNet18',
 'Hybrid/HPVQL',
 'Hybrid/HQE',
 'Hybrid/HQuanv',
 'Hybrid/HVQL',
 'Hybrid/HVQM',
 'Quantum/VQM']

Target groups:


['Classical/CNN',
 'Classical/MLP',
 'Classical/MLPMixer',
 'Classical/ResNet18',
 'Hybrid/HPVQL',
 'Hybrid/HQE',
 'Hybrid/HQuanv',
 'Hybrid/HVQL',
 'Hybrid/HVQM',
 'Quantum/VQM']

,epsilon,source_group,target_group,Transfer ASR,Transfer Robust Accuracy,Flip Rate,Mean Confidence Drop,Mean Margin Drop,mean,std,...,n_seeds,n_source_variants,n_target_variants,robust_accuracy,metric,comparison_mode,source_grain,target_grain,variant_aggregate_mode,architecture_aggregate_mode
0,0.0,Classical/CNN,Classical/MLP,0.0,0.9903,0.0,0.0,0.0,0.9903,0.112011,...,2,4,7,0.9903,robust_accuracy,inter_architecture,architecture,architecture,best_robust,mean
1,0.0,Classical/CNN,Classical/MLPMixer,0.0,0.9778,0.0,0.0,0.0,0.9778,0.000550,...,2,4,1,0.9778,robust_accuracy,inter_architecture,architecture,architecture,best_robust,mean
2,0.0,Classical/CNN,Classical/ResNet18,0.0,0.9924,0.0,0.0,0.0,0.9924,0.000200,...,2,4,1,0.9924,robust_accuracy,inter_architecture,architecture,architecture,best_robust,mean
3,0.0,Classical/CNN,Hybrid/HPVQL,0.0,0.9929,0.0,0.0,0.0,0.9929,0.000576,...,2,4,4,0.9929,robust_accuracy,inter_architecture,architecture,architecture,best_robust,mean
4,0.0,Classical/CNN,Hybrid/HQE,0.0,0.9847,0.0,0.0,0.0,0.9847,0.000800,...,2,4,1,0.9847,robust_accuracy,inter_architecture,architecture,architecture,best_robust,mean


## 4. Generate all figures and tables

CSV exports are kept flat under `outputs/attacksv2_transfer/curves/csv/`. Image exports are nested by family and role, for example `outputs/attacksv2_transfer/curves/images/classical/as_source/cnn_as_source__*.png` and `.../classical/as_target/cnn_as_target__*.png`. The exporter also writes one overview figure under `outputs/attacksv2_transfer/curves/images/overview/` with average source and target behavior.

In [15]:
all_curves, export_manifest = export_all_accuracy_curve_views(
    raw_df,
    CONFIG,
    OUT_DIR,
    include_self_attacks=INCLUDE_SELF_ATTACKS,
)

print(f"Combined curve rows: {len(all_curves):,}")
print(f"CSV folder: {OUT_DIR / 'csv'}")
print(f"Image folder: {OUT_DIR / 'images'}")
export_manifest

Exporting target curves: 100%|██████████| 10/10 [00:06<00:00,  1.54it/s]


Combined curve rows: 3,780
CSV folder: C:\Users\fwilh\Desktop\QB-PIER\outputs\attacksv2_transfer\curves\csv
Image folder: C:\Users\fwilh\Desktop\QB-PIER\outputs\attacksv2_transfer\curves\images


,mode,selected_group,n_rows,n_curves,file_stem,image_dir,png_path,pdf_path,overview_csv_path,data_csv_path,average_overview_csv_path
0,source,Classical/CNN,189,9,cnn_as_source__architecture_to_architecture__i...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,NaN,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...
1,source,Classical/MLP,189,9,mlp_as_source__architecture_to_architecture__i...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,NaN,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...
2,source,Classical/MLPMixer,189,9,mlpmixer_as_source__architecture_to_architectu...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,NaN,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...
3,source,Classical/ResNet18,189,9,resnet18_as_source__architecture_to_architectu...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,NaN,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...
4,source,Hybrid/HPVQL,189,9,hpvql_as_source__architecture_to_architecture_...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,NaN,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...
5,source,Hybrid/HQE,189,9,hqe_as_source__architecture_to_architecture__i...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,NaN,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...
6,source,Hybrid/HQuanv,189,9,hquanv_as_source__architecture_to_architecture...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,NaN,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...
7,source,Hybrid/HVQL,189,9,hvql_as_source__architecture_to_architecture__...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,NaN,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...
8,source,Hybrid/HVQM,189,9,hvqm_as_source__architecture_to_architecture__...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,NaN,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...
9,source,Quantum/VQM,189,9,vqm_as_source__architecture_to_architecture__i...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,NaN,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...,C:\Users\fwilh\Desktop\QB-PIER\outputs\attacks...
